## Interface

The objective of this code is to have a live interface that will display (for now):
- Actual position of the Thymio
- Arrival position
- Projected trajectory

Possible improvement ideas:
- Add obstacle delimitation
- Add security margins from obstacles
- Red screen when interruption due to hand/object appearance on screen

# Install

In [1]:
!pip install opencv-python

# Import

In [2]:
import time
import numpy as np
import cv2

from Vision_test import (MAP_WIDTH, MAP_HEIGHT, rectify_image, color_masks,
                         find_position, obstacle_map, occupancy_grid_from, a_star)

# Parameters

In [3]:
CAMERA_INDEX = 0           # index of the camera given to cv2.VideoCapture
CAMERA_WIDTH = 1280        # requested camera resolution (DSHOW default is often 640 x 480,
CAMERA_HEIGHT = 720        # too small to detect the QR codes reliably)
SNAPSHOT_PERIOD = 1.0      # seconds between two position updates
CALIBRATION_FRAMES = 10    # frames with the 4 QR codes needed to calibrate
MAP_SCALE = 4              # display zoom of the rectified map (205 x 155 px is small)
HEADING_LENGTH = 15        # length of the heading arrow, in map pixels

WINDOW_CAMERA = "Thymio - Camera Live"
WINDOW_MAP = "Thymio - Map"

# Corners of the rectified map, in map pixels (TL, TR, BR, BL)
MAP_CORNERS = np.array([
    [0, 0],
    [MAP_WIDTH - 1, 0],
    [MAP_WIDTH - 1, MAP_HEIGHT - 1],
    [0, MAP_HEIGHT - 1]
], dtype=np.float32)

# Camera calibration

Done **once at the start** (and again with the `c` key if the camera moves).

The live camera is displayed until the 4 QR codes have been detected on `CALIBRATION_FRAMES` frames.
For each of these frames, `rectify_image` (from `Vision_test.py`) gives the homography `H` (camera -> map);
the 4 corners of the map are projected back into the camera image and the median over the frames is kept,
which makes the calibration robust to a bad detection on one frame.

Returns:
- `H`: camera pixels -> map pixels
- `H_inv`: map pixels -> camera pixels (used to draw the Thymio on the live camera)
- `markers`: corners of the 4 QR codes in the camera image (used to remove them from the obstacle map)

In [4]:
def project_points(points, M):
    # Apply the homography M to a list of (x, y) points
    pts = np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, M).reshape(-1, 2)


def calibration_from_image(image):
    # Calibration from a single image (raises ValueError if the 4 QR codes are not found)
    _, markers, H = rectify_image(image)
    return H, np.linalg.inv(H), markers


def calibrate_camera(cap):
    # Live calibration. Returns (H, H_inv, markers), or None if 'q' is pressed.
    corners_list = []
    markers = None

    while len(corners_list) < CALIBRATION_FRAMES:
        ret, frame = cap.read()
        if not ret:
            raise RuntimeError("Unable to read a frame from the camera")

        display = frame.copy()
        try:
            _, frame_markers, H = rectify_image(frame)
            corners_list.append(project_points(MAP_CORNERS, np.linalg.inv(H)))
            markers = frame_markers
            for qr in frame_markers:
                cv2.polylines(display, [qr.astype(np.int32)], True, (0, 255, 0), 2)
            status, color = f"Calibration: {len(corners_list)}/{CALIBRATION_FRAMES}", (0, 255, 0)
        except ValueError as e:
            status, color = f"Calibration: {e}", (0, 0, 255)

        cv2.putText(display, status, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        cv2.putText(display, "q: quit", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.imshow(WINDOW_CAMERA, display)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            return None

    # Median of the map corners seen by the camera -> final homography
    corners = np.median(np.array(corners_list), axis=0).astype(np.float32)
    H = cv2.getPerspectiveTransform(corners, MAP_CORNERS)
    return H, np.linalg.inv(H), markers

# Thymio localisation

Done on a snapshot every `SNAPSHOT_PERIOD` seconds, **with the homography of the calibration**
(the QR codes are not searched again, so the Thymio can hide one of them).

Returns a dictionary with:
- `thymio`: `(x, y)` position of the Thymio in map pixels (middle of the red and green points), `None` if not detected
- `theta`: angle of the Thymio in radians (red point -> green point), same convention as `Vision_test.py`
- `goal`: `(x, y)` position of the blue point, `None` if not detected
- `path`: list of `(x, y)` points of the A* path from the Thymio to the goal, `None` if no path
- `rectified`, `obstacles`: rectified image and obstacle mask (for the display)

In [5]:
def locate_thymio(frame, H, markers):
    rectified = cv2.warpPerspective(frame, H, (MAP_WIDTH, MAP_HEIGHT))

    mask_red, mask_green, mask_blue = color_masks(rectified)
    red = find_position(mask_red)
    green = find_position(mask_green)
    blue = find_position(mask_blue)

    color_mask = cv2.bitwise_or(cv2.bitwise_or(mask_red, mask_green), mask_blue)
    obstacles = obstacle_map(rectified, color_mask, markers, H)

    state = {"rectified": rectified, "obstacles": obstacles,
             "thymio": None, "theta": None, "goal": blue, "path": None}

    if red is None or green is None:
        return state

    x = (red[0] + green[0]) / 2
    y = (red[1] + green[1]) / 2
    state["thymio"] = (x, y)
    state["theta"] = float(np.arctan2(green[1] - red[1], green[0] - red[0]))

    if blue is not None:
        grid = occupancy_grid_from(obstacles)
        start = (int(round(y)), int(round(x)))   # grid cells are (row, col) = (y, x)
        goal = (blue[1], blue[0])
        grid[start] = False
        grid[goal] = False
        path = a_star(grid, start, goal)
        if path is not None:
            state["path"] = [(col, row) for row, col in path]

    return state

# Data display

Two windows:
- **Camera**: live camera with the calibrated area, the Thymio (red point + heading), the goal (blue) and the A* path,
  projected from the map into the camera image with `H_inv`
- **Map**: rectified top view of the last snapshot, with obstacles in grey

In [6]:
def to_int(p):
    return (int(round(p[0])), int(round(p[1])))


def heading_point(state):
    x, y = state["thymio"]
    return (x + HEADING_LENGTH * np.cos(state["theta"]),
            y + HEADING_LENGTH * np.sin(state["theta"]))


def draw_camera(frame, state, H_inv, stale):
    # Calibrated area
    area = project_points(MAP_CORNERS, H_inv).astype(np.int32)
    cv2.polylines(frame, [area], True, (255, 255, 0), 2)

    if state is None:
        return frame

    if state["path"] is not None:
        path = project_points(state["path"], H_inv).astype(np.int32)
        cv2.polylines(frame, [path], False, (0, 255, 255), 2)

    if state["goal"] is not None:
        cv2.circle(frame, to_int(project_points([state["goal"]], H_inv)[0]), 8, (255, 0, 0), -1)

    if state["thymio"] is not None:
        thymio, head = project_points([state["thymio"], heading_point(state)], H_inv)
        color = (0, 0, 255) if not stale else (128, 128, 128)
        cv2.arrowedLine(frame, to_int(thymio), to_int(head), color, 2, tipLength=0.3)
        cv2.circle(frame, to_int(thymio), 8, color, -1)

        x, y = state["thymio"]
        cv2.putText(frame, f"Position: ({x:.1f}, {y:.1f}) px", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f"Theta: {np.degrees(state['theta']):.1f} deg", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    if stale:
        cv2.putText(frame, "Thymio not detected (last known position)", (10, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    return frame


def draw_map(state, stale):
    s = MAP_SCALE
    img = cv2.resize(state["rectified"], (MAP_WIDTH * s, MAP_HEIGHT * s),
                     interpolation=cv2.INTER_NEAREST)

    # Obstacles in grey
    obstacles = cv2.resize(state["obstacles"].astype(np.uint8), (MAP_WIDTH * s, MAP_HEIGHT * s),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
    img[obstacles] = (0.5 * img[obstacles] + 0.5 * np.array([128, 128, 128])).astype(np.uint8)

    def m(p):
        return (int(round(p[0] * s)), int(round(p[1] * s)))

    if state["path"] is not None:
        pts = np.array([m(p) for p in state["path"]], dtype=np.int32)
        cv2.polylines(img, [pts], False, (0, 255, 255), 2)

    if state["goal"] is not None:
        cv2.circle(img, m(state["goal"]), 8, (255, 0, 0), -1)

    if state["thymio"] is not None:
        color = (0, 0, 255) if not stale else (128, 128, 128)
        cv2.arrowedLine(img, m(state["thymio"]), m(heading_point(state)), color, 2, tipLength=0.3)
        cv2.circle(img, m(state["thymio"]), 8, color, -1)

    return img

# Test on a static image (without camera)

Calibration + localisation on `ThymioSetup4.jfif`, to check the pipeline without the camera.

In [7]:
image = cv2.imread("ThymioSetup4.jfif")
H, H_inv, markers = calibration_from_image(image)
state = locate_thymio(image, H, markers)
print("Thymio:", state["thymio"], "theta:", state["theta"], "goal:", state["goal"])

cv2.imshow(WINDOW_CAMERA, draw_camera(image.copy(), state, H_inv, stale=False))
cv2.imshow(WINDOW_MAP, draw_map(state, stale=False))
cv2.waitKey(0)
cv2.destroyAllWindows()

Thymio: (102.0, 104.0) theta: -1.5707963267948966 goal: (101, 14)


# Live interface

1. The camera turns on and calibrates itself (show the 4 QR codes)
2. Every `SNAPSHOT_PERIOD` seconds, a snapshot is taken and the Thymio is located on the map
3. Between two snapshots, the live camera is displayed with the last known position

Keys: `q` to quit, `c` to recalibrate the camera.

In [ ]:
cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)

if not cap.isOpened():
    print("Error: unable to open the camera")
else:
    try:
        # The camera may not support the requested resolution: print the real one
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(f"Camera activated! Resolution: {width} x {height}")
        print("Calibration... (show the 4 QR codes, q to quit)")
        calibration = calibrate_camera(cap)

        if calibration is None:
            print("Calibration aborted.")
        else:
            H, H_inv, markers = calibration
            print("Calibration done. q: quit, c: recalibrate")

            state = None          # last snapshot analysis
            last_pose = None      # last detected (x, y, theta)
            last_snapshot = 0.0

            while True:
                ret, frame = cap.read()
                if not ret:
                    print("Unable to read a frame from the camera")
                    break

                # Snapshot every SNAPSHOT_PERIOD seconds
                now = time.time()
                if now - last_snapshot >= SNAPSHOT_PERIOD:
                    last_snapshot = now
                    state = locate_thymio(frame, H, markers)
                    stale = state["thymio"] is None
                    if stale and last_pose is not None:
                        # Keep displaying the last known position
                        state["thymio"], state["theta"] = last_pose[:2], last_pose[2]
                    elif not stale:
                        last_pose = (*state["thymio"], state["theta"])
                    cv2.imshow(WINDOW_MAP, draw_map(state, stale))

                cv2.imshow(WINDOW_CAMERA, draw_camera(frame, state, H_inv, state is not None and stale))

                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'):
                    break
                if key == ord('c'):
                    calibration = calibrate_camera(cap)
                    if calibration is None:
                        break
                    H, H_inv, markers = calibration
                    print("Recalibration done.")
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Camera closed.")

Camera activated! Resolution: 1280 x 720
Calibration... (show the 4 QR codes, q to quit)
Calibration done. q: quit, c: recalibrate
